In [9]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import json
import time
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

In [22]:
# notebooks 폴더에서 실행한다고 가정
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

load_dotenv(PROJECT_ROOT / ".env")

GARAK_AUCTION_PASSWORD = os.getenv("GARAK_AUCTION_PASSWORD")

BASE_URL = "https://www.garak.co.kr/homepage/publicdata/dataJsonOpen.do"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "garak" / "auction"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트:", PROJECT_ROOT)
print("저장 위치:", RAW_DIR)
print("비밀번호 로드:", GARAK_AUCTION_PASSWORD is not None)

프로젝트: d:\CODE\proj2\5pl
저장 위치: d:\CODE\proj2\5pl\data\raw\garak\auction
비밀번호 로드: True


In [23]:
BUBINS = {
    "서울청과": "11000101",
    "농협(공)": "11000102",
    "중앙청과": "11000103",
    "동부팜청과": "11000104",
    "한국청과": "11000105",
    "대아청과": "11000106",
}

In [24]:
ITEMS = [
    "사과",
    "배",
    "대추",
]

In [33]:
def fetch_auction_page(
    date,
    item,
    bubin,
    page=1,
    page_size=100,
    origin=""
):
    params = {
        "id": "11116",
        "passwd": GARAK_AUCTION_PASSWORD,
        "dataid": "data12",
        "pagesize": page_size,
        "pageidx": page,
        "portal.templet": "false",
        "s_date": date,
        "s_bubin": bubin,
        "s_pummok": item,
        "s_sangi": origin,
    }

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=30
        )

        response.raise_for_status()

    except requests.RequestException as e:
        print(f"요청 오류 | {date} | {item} | page {page}")
        print(e)
        return []

    try:
        data = response.json()

    except requests.exceptions.JSONDecodeError:
        # 거래 없는 날 등에 JSON이 아닌 응답이 오는 경우
        return []

    return data.get("resultData", [])

In [34]:
test_records = fetch_auction_page(
    date="20220103",
    item="사과",
    bubin="11000101",
    page=1,
    page_size=10
)

In [35]:
test_df = pd.DataFrame(test_records)

test_df

,ADJ_DT,DDD,PPRICE,PUM_NAME_IMSI,CORP_NM,UUN,ROWNO,PUMMOK,QTY,PUMJONG,INJUNG_GUBUN,SSANGI
0,20220103,특(1등),60000,[사과]기꾸,서울청과,10kg,1,사과,1,기꾸,일반,충북 영동군
1,20220103,특(1등),45000,[사과]기꾸,서울청과,10kg,2,사과,2,기꾸,일반,충북 영동군
2,20220103,특(1등),43000,[사과]기꾸,서울청과,10kg,3,사과,4,기꾸,일반,충북 영동군
3,20220103,특(1등),36000,[사과]기꾸,서울청과,10kg,4,사과,9,기꾸,일반,충북 영동군
4,20220103,특(1등),35000,[사과]기꾸,서울청과,10kg,5,사과,3,기꾸,일반,충북 영동군
5,20220103,특(1등),35000,[사과]기꾸,서울청과,10kg,6,사과,23,기꾸,일반,충북 영동군
6,20220103,특(1등),35000,[사과]기꾸,서울청과,10kg,7,사과,9,기꾸,일반,충북 영동군
7,20220103,특(1등),30000,[사과]기꾸,서울청과,10kg,8,사과,7,기꾸,일반,충북 영동군
8,20220103,특(1등),30000,[사과]기꾸,서울청과,10kg,9,사과,16,기꾸,일반,충북 영동군
9,20220103,특(1등),27000,[사과]기꾸,서울청과,10kg,10,사과,14,기꾸,일반,충북 영동군


In [38]:
test_df["PUMMOK"].value_counts()

def fetch_all_pages_for_item(
    date,
    item,
    bubin,
    page_size=100,
    sleep_sec=0.2,
    max_pages=1000
):
    all_records = []

    for page in range(1, max_pages + 1):

        records = fetch_auction_page(
            date=date,
            item=item,
            bubin=bubin,
            page=page,
            page_size=page_size
        )

        if not records:
            break

        # '사과' 검색에 '사과대추' 등이 섞이는 문제 제거
        exact_records = [
            record
            for record in records
            if record.get("PUMMOK") == item
        ]

        all_records.extend(exact_records)

        # 페이지에 page_size보다 적게 왔다면 마지막 페이지
        if len(records) < page_size:
            break

        time.sleep(sleep_sec)

    return all_records

In [29]:
def fetch_all_pages_for_date(
    date,
    item,
    page_size=100,
    bubin="11000101",
    sleep_sec=0.2,
    max_pages=1000
):
    all_records = []
    previous_signature = None

    for page in range(1, max_pages + 1):
        records = fetch_auction_page(
            date=date,
            item=item,
            page=page,
            page_size=page_size,
            bubin=bubin
        )

        # 데이터가 없으면 종료
        if not records:
            print(f"{date} | page {page}: 데이터 없음 → 종료")
            break

        # 같은 페이지가 반복되는 API 오류 방지
        signature = (
            len(records),
            json.dumps(
                records[0],
                ensure_ascii=False,
                sort_keys=True
            ),
            json.dumps(
                records[-1],
                ensure_ascii=False,
                sort_keys=True
            )
        )

        if signature == previous_signature:
            print(f"{date} | page {page}: 이전 페이지와 동일 → 종료")
            break

        previous_signature = signature
        all_records.extend(records)

        print(
            f"{date} | page {page} | "
            f"{len(records)}건 | 누적 {len(all_records)}건"
        )

        # 핵심: 페이지 크기보다 적으면 마지막 페이지
        if len(records) < page_size:
            print(f"{date} | page {page}: 마지막 페이지 → 종료")
            break

        time.sleep(sleep_sec)

    return all_records

In [39]:
def fetch_one_day(
    date,
    items,
    bubins,
    page_size=100,
    sleep_sec=0.2
):
    day_records = []

    for corp_name, corp_code in bubins.items():

        print(f"\n[{date}] {corp_name}")

        for item in items:

            records = fetch_all_pages_for_item(
                date=date,
                item=item,
                bubin=corp_code,
                page_size=page_size,
                sleep_sec=sleep_sec
            )

            if records:
                day_records.extend(records)

                print(
                    f"  {item:<10} "
                    f"{len(records):>5}건"
                )

    return day_records

In [40]:
def make_date_list(start_date, end_date):

    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")

    dates = []

    current = start

    while current <= end:
        dates.append(current.strftime("%Y%m%d"))
        current += timedelta(days=1)

    return dates

In [41]:
test_records = fetch_one_day(
    date="20220103",
    items=ITEMS,
    bubins=BUBINS
)

print("\n총 수집 건수:", len(test_records))


[20220103] 서울청과
  사과           595건
  배            388건

[20220103] 농협(공)
  사과           226건
  배            142건

[20220103] 중앙청과
  사과           955건
  배            433건

[20220103] 동부팜청과
  사과           309건
  배             58건

[20220103] 한국청과
  사과           155건
  배             79건

[20220103] 대아청과

총 수집 건수: 3340
